In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
from matplotlib.colors import SymLogNorm
import yt
yt.set_log_level('error')
import os
from dotenv import dotenv_values
import glob
import pandas as pd
import re
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.stats import linregress


In [ ]:
kmax_start=80
kmax_end=120
kmax_inc=10
kmax_arr = np.arange(kmax_start, kmax_end + 1, kmax_inc)

In [ ]:
def read_in_bar_data(run_dir, kmax):
    f_dir = os.path.join(run_dir, f"filtered_9000_0_{kmax}")
    files = sorted(glob.glob(f_dir))
    # print(files)
    ts = yt.DatasetSeries(files)
    return ts

# def round_to_6_digits(val):
#     # Calculate how many integer digits there are
#     int_len = len(str(abs(int(val))))
#     # Round to the remaining number of decimal places
#     return round(val, 6 - int_len)
    
# def read_in_prime_data(run_dir, kmax):
#     f_dir = os.path.join(run_dir, f"filtered_9000_{round_to_6_digits(np.sqrt(kmax**2 + .5))}_{100*kmax}")
#     print(f_dir)
#     files = sorted(glob.glob(f_dir))
#     print(files)
#     ts = yt.DatasetSeries(files)
#     return ts
    
f_bar_list = []
# f_prime_list = []
N_seeds = 1
for i in range(N_seeds):
    f_bar_i_arr = []
    # f_prime_i_arr = []
    # run_dir_i = f"Kolmogorov_WENO_512_1em4_cell_depth_1em5_F_0_4em1_n_WN_10_dt_9em4_seed_{i+1}00"
    run_dir_i = f"Kolmogorov_WENO_512_1em4_cell_depth_1em5_F_0_1em1_n_WN_3_dt_3em4_seed_{i+1}00"
    # run_dir_i = f"Kolmogorov_WENO_512_1em4_cell_depth_1em5_F_0_1em1_n_WN_3_dt_3em4_stoch_flux_0_seed_{i+1}00"
    for kmax in kmax_arr:
        f_bar_i_kmax = read_in_bar_data(run_dir_i, kmax)
        # f_prime_i_kmax = read_in_prime_data(run_dir_i, kmax)
        
        f_bar_i_arr.append(f_bar_i_kmax)
        # f_prime_i_arr.append(f_prime_i_kmax)
        
    f_bar_list.append(f_bar_i_arr)
    # f_prime_list.append(f_prime_i_arr)


In [ ]:
f_bar_list[0][0][0].field_list

In [ ]:
def plot_field(field, label = "", title = ""):
    plt.figure(figsize=(8, 6), dpi=150)
    extent = [0, 1, 0, 1]

    im = plt.imshow(field, origin='lower', extent=extent, cmap='magma')
    # im = plt.imshow(trace_2d.T, origin='lower', extent=extent, cmap='magma', 
    #                 norm=LogNorm(vmin=vmin, vmax=vmax))
    
    plt.colorbar(im, label=label)

    # time = ds_final.current_time.v
    
    plt.title(title)
    
    plt.xlabel('$x$')
    plt.ylabel('$y$')
    plt.show()

In [ ]:
def outer_v(vel):
    u = vel[0, :, :]
    v = vel[1, :, :]

    uu = u * u
    uv = u * v
    vv = v * v

    return np.array([[uu, uv], [uv, vv]])

def turb_stress_tensor(uu_bar, vv_bar, uv_bar, vel_bar):
    vv_tensor_bar = np.array([[uu_bar, uv_bar],
                              [uv_bar, vv_bar]])
    # print(f"vel_bar.shape: {vel_bar.shape}", flush = True)
    
    return vv_tensor_bar - outer_v(vel_bar)

Nx, Ny = 512, 512
tau_arr = []
S_arr = []
for i in range(N_seeds):
    tau_arr_seeds = []
    S_arr_seeds = []
    for k, kmax in enumerate(kmax_arr):
        ds = f_bar_list[i][k][0]
        cg = ds.covering_grid(level=0, left_edge=ds.domain_left_edge, dims=[Nx, Ny, 1])
        uu_bar = np.array(cg['uu_filter'][:,:,0].v)
        vv_bar = np.array(cg['vv_filter'][:,:,0].v)
        uv_bar = np.array(cg['uv_filter'][:,:,0].v)
        vel_x_bar = np.array(cg['velx_filter'][:,:,0].v)
        vel_y_bar = np.array(cg['vely_filter'][:,:,0].v)
        vel_bar = np.array([vel_x_bar, vel_y_bar])
        tau = turb_stress_tensor(uu_bar, vv_bar, uv_bar, vel_bar)

        S11 = np.array(cg['S11'][:,:,0].v)
        S12 = np.array(cg['S12'][:,:,0].v)
        S22 = np.array(cg['S22'][:,:,0].v)
        S_t = np.array([[S11, S12], [S12, S22]])
        if k < 1:
            plot_field(tau[0,0,:,:], label = r'$\tau$', title = f'Turbulent stress tensor for kmax = {kmax}')
            plot_field(S_t[0,0,:,:], label = r'$S_{00}$', title = f'Strain rate tensor for kmax = {kmax}')
        tau_arr_seeds.append(tau)
        S_arr_seeds.append(S_t)
    tau_arr.append(tau_arr_seeds)
    S_arr.append(S_arr_seeds)